In [8]:
pip install xgboost optuna scikit-learn pandas torch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install --upgrade pip

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [28]:
!pip install xgboost==1.6.2

Defaulting to user installation because normal site-packages is not writeable
  Using cached xgboost-1.6.2-py3-none-macosx_12_0_arm64.whl.metadata (1.8 kB)
Using cached xgboost-1.6.2-py3-none-macosx_12_0_arm64.whl (1.5 MB)
  Attempting uninstall: xgboost
    Found existing installation: xgboost 2.0.3
    Uninstalling xgboost-2.0.3:
      Successfully uninstalled xgboost-2.0.3


In [18]:
!pip install --force-reinstall --no-deps xgboost==2.0.3

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 6.1 MB/s  0:00:00m eta 0:00:01
  Attempting uninstall: xgboost
    Found existing installation: xgboost 1.6.2
    Uninstalling xgboost-1.6.2:
      Successfully uninstalled xgboost-1.6.2


In [36]:
"""
XGBoost Fraud Detection — Max AUC with Optuna tuning
=====================================================
Expects preprocessed CSVs from the pipeline:
  preprocessed/train.csv
  preprocessed/val.csv
  preprocessed/test.csv
  preprocessed/column_metadata.pkl

Usage:
  python xgboost_fraud.py            # tune + train + evaluate
  python xgboost_fraud.py --no-tune  # skip Optuna, use best_params if saved
"""

import argparse
import pickle
import time
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    roc_auc_score,
)

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR        = Path("preprocessed")
OUTPUT_DIR      = Path("models/xgboost")
N_TRIALS        = 60       # Optuna trials — increase for better tuning
EARLY_STOP      = 50       # early stopping rounds during tuning
FINAL_ROUNDS    = 2000     # max boosting rounds for final model
SEED            = 42
OPTUNA_DB       = "sqlite:///optuna_xgb.db"   # persists across runs

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load data ─────────────────────────────────────────────────────────────────
def load_data():
    print("Loading preprocessed data …")
    with open(DATA_DIR / "column_metadata.pkl", "rb") as f:
        meta = pickle.load(f)

    target = meta["target"]

    train = pd.read_csv(DATA_DIR / "train.csv")
    val   = pd.read_csv(DATA_DIR / "val.csv")
    test  = pd.read_csv(DATA_DIR / "test.csv")

    feature_cols = [c for c in train.columns if c != target]

    X_train, y_train = train[feature_cols].values, train[target].values
    X_val,   y_val   = val[feature_cols].values,   val[target].values
    X_test,  y_test  = test[feature_cols].values,  test[target].values

    # Class imbalance ratio — used in scale_pos_weight
    neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
    scale_pos_weight = neg / pos
    print(f"  Train: {len(y_train):,}  |  fraud rate: {y_train.mean():.3%}")
    print(f"  Val  : {len(y_val):,}  |  fraud rate: {y_val.mean():.3%}")
    print(f"  scale_pos_weight = {scale_pos_weight:.1f}")

    # XGBoost DMatrix format
    dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=list(feature_cols))
    dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=list(feature_cols))
    dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=list(feature_cols))

    return dtrain, dval, dtest, y_val, y_test, scale_pos_weight, feature_cols


# ── Optuna objective ──────────────────────────────────────────────────────────
def make_objective(dtrain, dval, y_val, scale_pos_weight):
    def objective(trial):
        params = {
            "objective":        "binary:logistic",
            "eval_metric":      "auc",
            "tree_method":      "hist",          # fast on CPU & Apple Silicon
            "seed":             SEED,
            "verbosity":        0,

            # Imbalance
            "scale_pos_weight": scale_pos_weight,

            # ── Parameters Optuna tunes ──────────────────────────────────────
            "n_estimators":     trial.suggest_int("n_estimators", 300, 1500),
            "max_depth":        trial.suggest_int("max_depth", 4, 10),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
            "colsample_bylevel":trial.suggest_float("colsample_bylevel", 0.4, 1.0),
            "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
            "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "max_delta_step":   trial.suggest_int("max_delta_step", 0, 10),
        }

        n_estimators = params.pop("n_estimators")

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=n_estimators,
            evals=[(dval, "val")],
            early_stopping_rounds=EARLY_STOP,
            verbose_eval=False,
        )

        preds = model.predict(dval)
        return roc_auc_score(y_val, preds)

    return objective


# ── Train final model ─────────────────────────────────────────────────────────
def train_final(best_params, dtrain, dval, scale_pos_weight):
    print("\nTraining final model with best params …")
    params = {
        "objective":        "binary:logistic",
        "eval_metric":      ["auc", "aucpr", "logloss"],
        "tree_method":      "hist",
        "seed":             SEED,
        "verbosity":        1,
        "scale_pos_weight": scale_pos_weight,
        **{k: v for k, v in best_params.items() if k != "n_estimators"},
    }
    n_est = best_params.get("n_estimators", FINAL_ROUNDS)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=int(n_est * 1.1),   # slight boost over tuning rounds
        evals=[(dtrain, "train"), (dval, "val")],
        early_stopping_rounds=EARLY_STOP,
        verbose_eval=50,
    )
    return model


# ── Evaluate ──────────────────────────────────────────────────────────────────
def evaluate(model, dmat, y_true, split_name):
    probs = model.predict(dmat)
    auc   = roc_auc_score(y_true, probs)
    ap    = average_precision_score(y_true, probs)

    # Find threshold maximising F1 on this split
    from sklearn.metrics import f1_score
    thresholds = np.linspace(0.1, 0.9, 81)
    f1s        = [f1_score(y_true, probs >= t, zero_division=0) for t in thresholds]
    best_t     = thresholds[np.argmax(f1s)]
    preds      = (probs >= best_t).astype(int)

    print(f"\n{'─'*50}")
    print(f"  {split_name.upper()} RESULTS")
    print(f"  ROC-AUC  : {auc:.4f}")
    print(f"  PR-AUC   : {ap:.4f}")
    print(f"  Best threshold (F1): {best_t:.2f}")
    print(classification_report(y_true, preds, target_names=["legit", "fraud"]))

    return {"auc": auc, "pr_auc": ap, "threshold": best_t, "probs": probs}


# ── Feature importance ────────────────────────────────────────────────────────
def show_feature_importance(model, top_n=20):
    scores = model.get_score(importance_type="gain")
    imp    = pd.Series(scores).sort_values(ascending=False).head(top_n)
    print(f"\nTop {top_n} features by gain:")
    print(imp.to_string())
    imp.to_csv(OUTPUT_DIR / "feature_importance.csv", header=["gain"])


# ── Main ──────────────────────────────────────────────────────────────────────
def main(tune: bool):
    dtrain, dval, dtest, y_val, y_test, spw, feature_cols = load_data()

    best_params_path = OUTPUT_DIR / "best_params.pkl"

    if tune:
        print(f"\nStarting Optuna study ({N_TRIALS} trials) …")
        sampler = optuna.samplers.TPESampler(seed=SEED)
        pruner  = optuna.pruners.MedianPruner(n_warmup_steps=10)
        study   = optuna.create_study(
            direction     = "maximize",
            sampler       = sampler,
            pruner        = pruner,
            study_name    = "xgb_fraud_auc",
            storage       = OPTUNA_DB,
            load_if_exists= True,
        )
        study.optimize(
            make_objective(dtrain, dval, y_val, spw),
            n_trials   = N_TRIALS,
            show_progress_bar = True,
        )
        best_params = study.best_params
        best_params["n_estimators"] = study.best_trial.last_step + 1
        print(f"\nBest val AUC : {study.best_value:.4f}")
        print(f"Best params  : {best_params}")
        with open(best_params_path, "wb") as f:
            pickle.dump(best_params, f)
    else:
        if not best_params_path.exists():
            raise FileNotFoundError(
                "No saved params found. Run with --tune first."
            )
        with open(best_params_path, "rb") as f:
            best_params = pickle.load(f)
        print(f"Loaded saved params: {best_params}")

    # ── Train final model ──────────────────────────────────────────────────
    t0    = time.time()
    model = train_final(best_params, dtrain, dval, spw)
    print(f"\nTraining time: {time.time()-t0:.1f}s")

    # ── Evaluate ───────────────────────────────────────────────────────────
    val_results  = evaluate(model, dval,  y_val,  "validation")
    test_results = evaluate(model, dtest, y_test, "test")

    # ── Feature importance ─────────────────────────────────────────────────
    show_feature_importance(model)

    # ── Save model ─────────────────────────────────────────────────────────
    model_path = OUTPUT_DIR / "xgb_fraud.ubj"
    model.save_model(model_path)
    print(f"\nModel saved → {model_path}")

    # Save test predictions
    pd.DataFrame({
        "fraud_prob": test_results["probs"],
        "pred":       (test_results["probs"] >= test_results["threshold"]).astype(int),
    }).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

    return model, val_results, test_results


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--no-tune", action="store_true", help="Skip Optuna tuning")
    args, unknown = parser.parse_known_args()

    main(tune=not args.no_tune)

Loading preprocessed data …
  Train: 472,443  |  fraud rate: 3.499%
  Val  : 29,516  |  fraud rate: 3.500%
  scale_pos_weight = 27.6


[I 2026-03-27 08:39:49,435] Using an existing study with name 'xgb_fraud_auc' instead of creating a new one.



Starting Optuna study (60 trials) …


Best trial: 15. Best value: 0.972422:   2%|▏         | 1/60 [00:43<42:18, 43.03s/it]

[I 2026-03-27 08:40:32,468] Trial 15 finished with value: 0.9724221295500086 and parameters: {'n_estimators': 954, 'max_depth': 9, 'min_child_weight': 9, 'learning_rate': 0.0466740590245877, 'subsample': 0.6786809959450116, 'colsample_bytree': 0.8141482441414871, 'colsample_bylevel': 0.5310188048290371, 'gamma': 3.189858776055304, 'reg_alpha': 3.3316309468938208, 'reg_lambda': 1.3488906627463655e-07, 'max_delta_step': 9}. Best is trial 15 with value: 0.9724221295500086.


...

Best trial: 68. Best value: 0.974353:  98%|█████████▊| 59/60 [28:34<00:38, 38.39s/it]

[I 2026-03-27 09:08:24,183] Trial 73 finished with value: 0.9717715997032113 and parameters: {'n_estimators': 943, 'max_depth': 9, 'min_child_weight': 2, 'learning_rate': 0.038980642410265366, 'subsample': 0.8715186882630275, 'colsample_bytree': 0.45454212887781903, 'colsample_bylevel': 0.7153609640641092, 'gamma': 2.316408047395484, 'reg_alpha': 4.078275393752048e-08, 'reg_lambda': 2.160119651988172, 'max_delta_step': 4}. Best is trial 68 with value: 0.9743530549412484.


Best trial: 68. Best value: 0.974353: 100%|██████████| 60/60 [29:21<00:00, 29.35s/it]

[I 2026-03-27 09:09:10,524] Trial 74 finished with value: 0.9724574761209275 and parameters: {'n_estimators': 1074, 'max_depth': 9, 'min_child_weight': 3, 'learning_rate': 0.04984444611975243, 'subsample': 0.9124168369387267, 'colsample_bytree': 0.4015668141488321, 'colsample_bylevel': 0.7190634057812046, 'gamma': 1.8722172547078397, 'reg_alpha': 1.2904805701570458e-06, 'reg_lambda': 9.75925033655839, 'max_delta_step': 1}. Best is trial 68 with value: 0.9743530549412484.


TypeError: unsupported operand type(s) for +: 'NoneType' and 'int'

In [20]:
import sys, platform
print(sys.version)
print(platform.machine())
print(sys.executable)

3.9.6 (default, Jan  9 2026, 11:03:41) 
[Clang 17.0.0 (clang-1700.6.4.2)]
arm64
/Library/Developer/CommandLineTools/usr/bin/python3


In [21]:
import subprocess, sys

commands = [
    "sudo chown -R anbu /opt/homebrew",
    "brew install python@3.11",
    "/opt/homebrew/bin/pip3.11 install xgboost optuna scikit-learn pandas torch notebook",
]

for cmd in commands:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout[-500:] if result.stdout else "")
    print(result.stderr[-300:] if result.stderr else "")


sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: a password is required


rew/share/info /opt/homebrew/share/locale /opt/homebrew/share/man /opt/homebrew/share/man/man1 /opt/homebrew/share/man/man3 /opt/homebrew/share/man/man5 /opt/homebrew/share/man/man7 /opt/homebrew/share/zsh /opt/homebrew/share/zsh/site-functions /opt/homebrew/var/homebrew/locks /opt/homebrew/var/log


/bin/sh: /opt/homebrew/bin/pip3.11: No such file or directory



In [24]:
"""
DNN Fraud Detection — Embedding + MLP (PyTorch)
================================================
Architecture:
  • Categorical cols  → individual Embedding layers → concatenated
  • Numerical cols    → passed directly (already scaled)
  • Concat            → BatchNorm → MLP with residual skip connections
                     → Sigmoid output

Device priority: CUDA → MPS (Apple Silicon) → CPU

Expects preprocessed outputs from the pipeline:
  preprocessed/train.csv
  preprocessed/val.csv
  preprocessed/test.csv
  preprocessed/column_metadata.pkl

Usage:
  python dnn_fraud.py              # train with default hyperparams
  python dnn_fraud.py --tune       # Optuna sweep over arch hyperparams
"""

import argparse
import pickle
import time
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR      = Path("preprocessed")
OUTPUT_DIR    = Path("models/dnn")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED          = 42
BATCH_SIZE    = 4096
MAX_EPOCHS    = 50
PATIENCE      = 7        # early stopping patience on val AUC
N_TUNE_TRIALS = 30       # Optuna trials when --tune is passed

torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
def get_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
    elif torch.backends.mps.is_available():
        dev = torch.device("mps")
    else:
        dev = torch.device("cpu")
    print(f"Using device: {dev}")
    return dev

DEVICE = get_device()


# ── Dataset ───────────────────────────────────────────────────────────────────
class FraudDataset(Dataset):
    """
    Splits columns into:
      cat_data  — integer indices for embedding lookup  (LongTensor)
      num_data  — scaled floats                          (FloatTensor)
      labels    — 0/1                                    (FloatTensor)
    """
    def __init__(self, df, cat_cols, num_cols, target):
        self.cat = torch.tensor(df[cat_cols].values, dtype=torch.long)
        self.num = torch.tensor(df[num_cols].values, dtype=torch.float32)
        self.y   = torch.tensor(df[target].values,   dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.cat[idx], self.num[idx], self.y[idx]


def make_sampler(y_train_np):
    """WeightedRandomSampler → each mini-batch ~50/50 fraud/legit."""
    fraud_rate = y_train_np.mean()
    w_fraud    = 1.0 / fraud_rate
    w_legit    = 1.0 / (1.0 - fraud_rate)
    weights    = np.where(y_train_np == 1, w_fraud, w_legit)
    return WeightedRandomSampler(
        weights     = torch.DoubleTensor(weights),
        num_samples = len(weights),
        replacement = True,
    )


# ── Model ─────────────────────────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    """Two-layer residual block with BatchNorm, GELU, and dropout."""
    def __init__(self, dim, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(x + self.block(x))


class EmbeddingMLP(nn.Module):
    """
    Embedding + MLP fraud classifier.

    Each categorical column gets its own Embedding layer.
    Embedding dim uses the rule-of-thumb: min(embed_dim, max(4, vocab//2)).
    All embeddings + numerical features are concatenated, then passed through
    an input projection and a stack of residual blocks.

    Parameters
    ----------
    vocab_sizes  : dict  {col_name: vocab_size}
    num_features : int   number of numerical input features
    embed_dim    : int   max embedding dimension per column
    hidden_dim   : int   MLP width
    n_blocks     : int   number of residual blocks
    dropout      : float dropout rate
    """
    def __init__(
        self,
        vocab_sizes : dict,
        num_features: int,
        embed_dim   : int   = 16,
        hidden_dim  : int   = 256,
        n_blocks    : int   = 3,
        dropout     : float = 0.3,
    ):
        super().__init__()
        self.cat_cols = list(vocab_sizes.keys())

        # Per-column embedding dims
        col_dims = {
            col: min(embed_dim, max(4, vocab_sizes[col] // 2))
            for col in self.cat_cols
        }
        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_sizes[col], col_dims[col])
            for col in self.cat_cols
        ])

        embed_total = sum(col_dims.values())
        input_dim   = embed_total + num_features

        # Input projection
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Residual blocks
        self.blocks = nn.ModuleList([
            ResidualBlock(hidden_dim, dropout) for _ in range(n_blocks)
        ])

        # Output head
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(hidden_dim // 2, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)

    def forward(self, cat, num):
        embeds = [emb(cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(embeds + [num], dim=1)
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x).squeeze(1)   # raw logits, shape (B,)


# ── Focal Loss ────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    Focuses training on hard, misclassified examples.
    Helps with the heavy class imbalance in fraud datasets.
    """
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce  = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        prob = torch.sigmoid(logits)
        pt   = torch.where(targets == 1, prob, 1.0 - prob)
        at   = torch.where(
            targets == 1,
            torch.full_like(pt, self.alpha),
            torch.full_like(pt, 1.0 - self.alpha),
        )
        return (at * (1.0 - pt) ** self.gamma * bce).mean()


# ── Train / eval loops ────────────────────────────────────────────────────────
def train_epoch(model, loader, optimiser, criterion, device):
    model.train()
    total_loss = 0.0
    for cat, num, y in loader:
        cat, num, y = cat.to(device), num.to(device), y.to(device)
        optimiser.zero_grad()
        loss = criterion(model(cat, num), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_loader(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for cat, num, y in loader:
        probs = torch.sigmoid(model(cat.to(device), num.to(device))).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    return roc_auc_score(labels, probs), average_precision_score(labels, probs), probs, labels


# ── Full training run ─────────────────────────────────────────────────────────
def run_training(train_ds, val_ds, test_ds,
                 vocab_sizes, n_num, hparams, device, verbose=True):

    # pin_memory causes issues on MPS
    pin = device.type == "cuda"

    train_dl = DataLoader(
        train_ds, batch_size=BATCH_SIZE,
        sampler=make_sampler(train_ds.y.numpy()),
        num_workers=0, pin_memory=pin,
    )
    val_dl = DataLoader(val_ds,  batch_size=BATCH_SIZE * 2,
                        shuffle=False, num_workers=0)
    test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE * 2,
                         shuffle=False, num_workers=0)

    model = EmbeddingMLP(
        vocab_sizes  = vocab_sizes,
        num_features = n_num,
        embed_dim    = hparams["embed_dim"],
        hidden_dim   = hparams["hidden_dim"],
        n_blocks     = hparams["n_blocks"],
        dropout      = hparams["dropout"],
    ).to(device)

    if verbose:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Trainable parameters: {n_params:,}")

    criterion = FocalLoss(alpha=hparams["focal_alpha"], gamma=hparams["focal_gamma"])
    optimiser = torch.optim.AdamW(
        model.parameters(),
        lr=hparams["lr"], weight_decay=hparams["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimiser, T_max=MAX_EPOCHS, eta_min=1e-6
    )

    best_auc   = 0.0
    best_state = None
    patience_n = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        t0         = time.time()
        train_loss = train_epoch(model, train_dl, optimiser, criterion, device)
        val_auc, val_pr, _, _ = eval_loader(model, val_dl, device)
        scheduler.step()

        improved = val_auc > best_auc
        if improved:
            best_auc   = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_n = 0
        else:
            patience_n += 1

        if verbose:
            tick = " ✓" if improved else ""
            print(
                f"Epoch {epoch:03d} | loss {train_loss:.4f} | "
                f"val AUC {val_auc:.4f} | val PR-AUC {val_pr:.4f} | "
                f"{time.time()-t0:.1f}s{tick}"
            )

        if patience_n >= PATIENCE:
            if verbose:
                print(f"Early stopping triggered at epoch {epoch}.")
            break

    # Restore best checkpoint and evaluate on test
    model.load_state_dict(best_state)
    test_auc, test_pr, test_probs, test_labels = eval_loader(model, test_dl, device)

    if verbose:
        print(f"\nBest val  AUC  : {best_auc:.4f}")
        print(f"Test      AUC  : {test_auc:.4f}")
        print(f"Test   PR-AUC  : {test_pr:.4f}")
        # Threshold sweep for best F1
        from sklearn.metrics import f1_score
        thresholds = np.linspace(0.05, 0.95, 91)
        f1s        = [f1_score(test_labels, test_probs >= t, zero_division=0)
                      for t in thresholds]
        best_t     = thresholds[np.argmax(f1s)]
        print(f"Best threshold (F1 on test): {best_t:.2f}")
        print(classification_report(
            test_labels, test_probs >= best_t,
            target_names=["legit", "fraud"],
        ))

    return model, best_auc, test_auc, test_probs


# ── Optuna sweep ──────────────────────────────────────────────────────────────
def tune(train_ds, val_ds, vocab_sizes, n_num, device):
    def objective(trial):
        hp = {
            "embed_dim"   : trial.suggest_categorical("embed_dim",  [8, 16, 32]),
            "hidden_dim"  : trial.suggest_categorical("hidden_dim", [128, 256, 512]),
            "n_blocks"    : trial.suggest_int("n_blocks", 2, 5),
            "dropout"     : trial.suggest_float("dropout", 0.1, 0.5),
            "lr"          : trial.suggest_float("lr", 1e-4, 3e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
            "focal_alpha" : trial.suggest_float("focal_alpha", 0.1, 0.5),
            "focal_gamma" : trial.suggest_float("focal_gamma", 1.0, 4.0),
        }
        _, val_auc, _, _ = run_training(
            train_ds, val_ds, val_ds,
            vocab_sizes, n_num, hp, device, verbose=False,
        )
        return val_auc

    study = optuna.create_study(
        direction     = "maximize",
        sampler       = optuna.samplers.TPESampler(seed=SEED),
        study_name    = "dnn_fraud_auc",
        storage       = "sqlite:///optuna_dnn.db",
        load_if_exists= True,
    )
    study.optimize(objective, n_trials=N_TUNE_TRIALS, show_progress_bar=True)
    print(f"\nBest val AUC : {study.best_value:.4f}")
    print(f"Best params  : {study.best_params}")
    return study.best_params


# ── Data loading ──────────────────────────────────────────────────────────────
def load_data():
    print("Loading preprocessed data …")
    with open(DATA_DIR / "column_metadata.pkl", "rb") as f:
        meta = pickle.load(f)

    target      = meta["target"]
    cat_cols    = meta["cat_cols"]
    vocab_sizes = meta["vocab_sizes"]

    train = pd.read_csv(DATA_DIR / "train.csv")
    val   = pd.read_csv(DATA_DIR / "val.csv")
    test  = pd.read_csv(DATA_DIR / "test.csv")

    # Filter to cols present in the dataframe (after feature engineering)
    cat_cols    = [c for c in cat_cols if c in train.columns]
    vocab_sizes = {c: vocab_sizes[c] for c in cat_cols}

    # Everything that is not a cat col or the target is treated as numerical
    num_cols = [c for c in train.columns if c not in cat_cols and c != target]

    print(f"  Cat features : {len(cat_cols)}")
    print(f"  Num features : {len(num_cols)}")
    print(f"  Train rows   : {len(train):,}  | fraud rate: {train[target].mean():.3%}")
    print(f"  Val rows     : {len(val):,}  | fraud rate: {val[target].mean():.3%}")

    train_ds = FraudDataset(train, cat_cols, num_cols, target)
    val_ds   = FraudDataset(val,   cat_cols, num_cols, target)
    test_ds  = FraudDataset(test,  cat_cols, num_cols, target)

    return train_ds, val_ds, test_ds, vocab_sizes, len(num_cols)


# ── Main ──────────────────────────────────────────────────────────────────────
DEFAULT_HPARAMS = {
    "embed_dim"   : 16,
    "hidden_dim"  : 256,
    "n_blocks"    : 3,
    "dropout"     : 0.3,
    "lr"          : 5e-4,
    "weight_decay": 1e-4,
    "focal_alpha" : 0.25,
    "focal_gamma" : 2.0,
}


def main(do_tune: bool):
    train_ds, val_ds, test_ds, vocab_sizes, n_num = load_data()

    if do_tune:
        print(f"\nStarting Optuna DNN sweep ({N_TUNE_TRIALS} trials) …")
        best_params = tune(train_ds, val_ds, vocab_sizes, n_num, DEVICE)
        hparams = {**DEFAULT_HPARAMS, **best_params}
    else:
        hparams = DEFAULT_HPARAMS
        print(f"\nUsing default hyperparams: {hparams}")

    print("\n── Training final model ─────────────────────────────────────────")
    model, best_val_auc, test_auc, test_probs = run_training(
        train_ds, val_ds, test_ds,
        vocab_sizes, n_num, hparams, DEVICE, verbose=True,
    )

    # Save model and predictions
    model_path = OUTPUT_DIR / "dnn_fraud.pt"
    torch.save({
        "model_state": model.state_dict(),
        "hparams"    : hparams,
        "vocab_sizes": vocab_sizes,
        "n_num"      : n_num,
    }, model_path)
    print(f"\nModel saved → {model_path}")

    pd.DataFrame({"fraud_prob": test_probs}).to_csv(
        OUTPUT_DIR / "test_predictions.csv", index=False
    )


DO_TUNE = False  # Set to True to run Optuna sweep first
main(do_tune=DO_TUNE)

Using device: mps
Loading preprocessed data …
  Cat features : 17
  Num features : 254
  Train rows   : 472,443  | fraud rate: 3.499%
  Val rows     : 29,516  | fraud rate: 3.500%

Using default hyperparams: {'embed_dim': 16, 'hidden_dim': 256, 'n_blocks': 3, 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.0001, 'focal_alpha': 0.25, 'focal_gamma': 2.0}

── Training final model ─────────────────────────────────────────
Trainable parameters: 759,581
Epoch 001 | loss 0.0898 | val AUC 0.8601 | val PR-AUC 0.3836 | 8.8s ✓
Epoch 002 | loss 0.0531 | val AUC 0.8797 | val PR-AUC 0.4359 | 5.4s ✓
Epoch 003 | loss 0.0469 | val AUC 0.9018 | val PR-AUC 0.4846 | 5.2s ✓
Epoch 004 | loss 0.0410 | val AUC 0.9137 | val PR-AUC 0.5328 | 5.6s ✓
Epoch 005 | loss 0.0376 | val AUC 0.9182 | val PR-AUC 0.5604 | 5.5s ✓
Epoch 006 | loss 0.0353 | val AUC 0.9209 | val PR-AUC 0.5797 | 5.6s ✓
Epoch 007 | loss 0.0335 | val AUC 0.9237 | val PR-AUC 0.5955 | 5.4s ✓
Epoch 008 | loss 0.0320 | val AUC 0.9261 | val PR-AUC 0.61

In [26]:
"""
DNN Fraud Detection — Embedding + MLP (PyTorch)
================================================
Architecture:
  • Categorical cols  → individual Embedding layers → concatenated
  • Numerical cols    → passed directly (already scaled)
  • Concat            → BatchNorm → MLP with TAPERED residual skip connections
                        (512 → 256 → 128) → Sigmoid output

Device priority: CUDA → MPS (Apple Silicon) → CPU

Expects preprocessed outputs from the pipeline:
  preprocessed/train.csv
  preprocessed/val.csv
  preprocessed/test.csv
  preprocessed/column_metadata.pkl

Usage:
  python dnn_fraud.py              # train with default hyperparams
  python dnn_fraud.py --tune       # Optuna sweep over arch hyperparams
"""

import argparse
import pickle
import time
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR      = Path("preprocessed")
OUTPUT_DIR    = Path("models/dnn")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED          = 42
BATCH_SIZE    = 4096
MAX_EPOCHS    = 50
PATIENCE      = 7        # early stopping patience on val AUC
N_TUNE_TRIALS = 30       # Optuna trials when --tune is passed

torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
def get_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
    elif torch.backends.mps.is_available():
        dev = torch.device("mps")
    else:
        dev = torch.device("cpu")
    print(f"Using device: {dev}")
    return dev

DEVICE = get_device()


# ── Dataset ───────────────────────────────────────────────────────────────────
class FraudDataset(Dataset):
    """
    Splits columns into:
      cat_data  — integer indices for embedding lookup  (LongTensor)
      num_data  — scaled floats                          (FloatTensor)
      labels    — 0/1                                    (FloatTensor)
    """
    def __init__(self, df, cat_cols, num_cols, target):
        self.cat = torch.tensor(df[cat_cols].values, dtype=torch.long)
        self.num = torch.tensor(df[num_cols].values, dtype=torch.float32)
        self.y   = torch.tensor(df[target].values,   dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.cat[idx], self.num[idx], self.y[idx]


def make_sampler(y_train_np):
    """WeightedRandomSampler → each mini-batch ~50/50 fraud/legit."""
    fraud_rate = y_train_np.mean()
    w_fraud    = 1.0 / fraud_rate
    w_legit    = 1.0 / (1.0 - fraud_rate)
    weights    = np.where(y_train_np == 1, w_fraud, w_legit)
    return WeightedRandomSampler(
        weights     = torch.DoubleTensor(weights),
        num_samples = len(weights),
        replacement = True,
    )


# ── Model ─────────────────────────────────────────────────────────────────────
class TaperedResidualBlock(nn.Module):
    """
    Residual block that projects from in_dim → out_dim.
    When in_dim != out_dim a linear shortcut aligns the skip connection,
    so gradients still flow cleanly at every width transition.
    """
    def __init__(self, in_dim: int, out_dim: int, dropout: float):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim, out_dim),
            nn.BatchNorm1d(out_dim),
        )
        # Shortcut: project only when widths differ
        self.shortcut = (
            nn.Linear(in_dim, out_dim, bias=False)
            if in_dim != out_dim else nn.Identity()
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.shortcut(x) + self.block(x))


class EmbeddingMLP(nn.Module):
    """
    Embedding + MLP fraud classifier with tapered residual blocks.

    Each categorical column gets its own Embedding layer.
    Embedding dim uses the rule-of-thumb: min(embed_dim, max(4, vocab//2)).
    All embeddings + numerical features are concatenated, projected to
    hidden_dim (= first block width), then passed through a stack of
    tapered residual blocks that halve the width each time.

    Block widths: hidden_dim → hidden_dim//2 → hidden_dim//4
    Default with hidden_dim=512: 512 → 256 → 128

    Parameters
    ----------
    vocab_sizes  : dict  {col_name: vocab_size}
    num_features : int   number of numerical input features
    embed_dim    : int   max embedding dimension per column
    hidden_dim   : int   width of the FIRST residual block (should be 512)
    n_blocks     : int   number of residual blocks (3 recommended for tapering)
    dropout      : float dropout rate
    """
    def __init__(
        self,
        vocab_sizes : dict,
        num_features: int,
        embed_dim   : int   = 16,
        hidden_dim  : int   = 512,
        n_blocks    : int   = 3,
        dropout     : float = 0.3,
    ):
        super().__init__()
        self.cat_cols = list(vocab_sizes.keys())

        # Per-column embedding dims
        col_dims = {
            col: min(embed_dim, max(4, vocab_sizes[col] // 2))
            for col in self.cat_cols
        }
        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_sizes[col], col_dims[col])
            for col in self.cat_cols
        ])

        embed_total = sum(col_dims.values())
        input_dim   = embed_total + num_features

        # Input projection → first block width
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Tapered residual blocks: each block halves the width
        # e.g. hidden_dim=512 → [512→256, 256→128, 128→64]
        # but we clamp so width never falls below 64
        dims = []
        w = hidden_dim
        for _ in range(n_blocks):
            next_w = max(w // 2, 64)
            dims.append((w, next_w))
            w = next_w

        self.blocks = nn.ModuleList([
            TaperedResidualBlock(in_d, out_d, dropout)
            for in_d, out_d in dims
        ])

        final_dim = dims[-1][1]   # width after last block

        # Output head
        self.head = nn.Sequential(
            nn.Linear(final_dim, final_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(final_dim // 2, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)

    def forward(self, cat, num):
        embeds = [emb(cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(embeds + [num], dim=1)
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x).squeeze(1)   # raw logits, shape (B,)


# ── Focal Loss ────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    Focuses training on hard, misclassified examples.
    Helps with the heavy class imbalance in fraud datasets.
    """
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce  = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        prob = torch.sigmoid(logits)
        pt   = torch.where(targets == 1, prob, 1.0 - prob)
        at   = torch.where(
            targets == 1,
            torch.full_like(pt, self.alpha),
            torch.full_like(pt, 1.0 - self.alpha),
        )
        return (at * (1.0 - pt) ** self.gamma * bce).mean()


# ── Train / eval loops ────────────────────────────────────────────────────────
def train_epoch(model, loader, optimiser, criterion, device):
    model.train()
    total_loss = 0.0
    for cat, num, y in loader:
        cat, num, y = cat.to(device), num.to(device), y.to(device)
        optimiser.zero_grad()
        loss = criterion(model(cat, num), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_loader(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for cat, num, y in loader:
        probs = torch.sigmoid(model(cat.to(device), num.to(device))).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    return roc_auc_score(labels, probs), average_precision_score(labels, probs), probs, labels


# ── Full training run ─────────────────────────────────────────────────────────
def run_training(train_ds, val_ds, test_ds,
                 vocab_sizes, n_num, hparams, device, verbose=True):

    # pin_memory causes issues on MPS
    pin = device.type == "cuda"

    train_dl = DataLoader(
        train_ds, batch_size=BATCH_SIZE,
        sampler=make_sampler(train_ds.y.numpy()),
        num_workers=0, pin_memory=pin,
    )
    val_dl = DataLoader(val_ds,  batch_size=BATCH_SIZE * 2,
                        shuffle=False, num_workers=0)
    test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE * 2,
                         shuffle=False, num_workers=0)

    model = EmbeddingMLP(
        vocab_sizes  = vocab_sizes,
        num_features = n_num,
        embed_dim    = hparams["embed_dim"],
        hidden_dim   = hparams["hidden_dim"],
        n_blocks     = hparams["n_blocks"],
        dropout      = hparams["dropout"],
    ).to(device)

    if verbose:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Trainable parameters: {n_params:,}")

    criterion = FocalLoss(alpha=hparams["focal_alpha"], gamma=hparams["focal_gamma"])
    optimiser = torch.optim.AdamW(
        model.parameters(),
        lr=hparams["lr"], weight_decay=hparams["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimiser, T_max=MAX_EPOCHS, eta_min=1e-6
    )

    best_auc   = 0.0
    best_state = None
    patience_n = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        t0         = time.time()
        train_loss = train_epoch(model, train_dl, optimiser, criterion, device)
        val_auc, val_pr, _, _ = eval_loader(model, val_dl, device)
        scheduler.step()

        improved = val_auc > best_auc
        if improved:
            best_auc   = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_n = 0
        else:
            patience_n += 1

        if verbose:
            tick = " ✓" if improved else ""
            print(
                f"Epoch {epoch:03d} | loss {train_loss:.4f} | "
                f"val AUC {val_auc:.4f} | val PR-AUC {val_pr:.4f} | "
                f"{time.time()-t0:.1f}s{tick}"
            )

        if patience_n >= PATIENCE:
            if verbose:
                print(f"Early stopping triggered at epoch {epoch}.")
            break

    # Restore best checkpoint and evaluate on test
    model.load_state_dict(best_state)
    test_auc, test_pr, test_probs, test_labels = eval_loader(model, test_dl, device)

    if verbose:
        print(f"\nBest val  AUC  : {best_auc:.4f}")
        print(f"Test      AUC  : {test_auc:.4f}")
        print(f"Test   PR-AUC  : {test_pr:.4f}")
        # Threshold sweep for best F1
        from sklearn.metrics import f1_score
        thresholds = np.linspace(0.05, 0.95, 91)
        f1s        = [f1_score(test_labels, test_probs >= t, zero_division=0)
                      for t in thresholds]
        best_t     = thresholds[np.argmax(f1s)]
        print(f"Best threshold (F1 on test): {best_t:.2f}")
        print(classification_report(
            test_labels, test_probs >= best_t,
            target_names=["legit", "fraud"],
        ))

    return model, best_auc, test_auc, test_probs


# ── Optuna sweep ──────────────────────────────────────────────────────────────
def tune(train_ds, val_ds, vocab_sizes, n_num, device):
    def objective(trial):
        hp = {
            "embed_dim"   : trial.suggest_categorical("embed_dim",  [8, 16, 32]),
            # hidden_dim is now the FIRST (widest) block; taper does the rest
            "hidden_dim"  : trial.suggest_categorical("hidden_dim", [256, 512, 1024]),
            "n_blocks"    : trial.suggest_int("n_blocks", 2, 5),
            "dropout"     : trial.suggest_float("dropout", 0.1, 0.5),
            "lr"          : trial.suggest_float("lr", 1e-4, 3e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
            "focal_alpha" : trial.suggest_float("focal_alpha", 0.1, 0.5),
            "focal_gamma" : trial.suggest_float("focal_gamma", 1.0, 4.0),
        }
        _, val_auc, _, _ = run_training(
            train_ds, val_ds, val_ds,
            vocab_sizes, n_num, hp, device, verbose=False,
        )
        return val_auc

    study = optuna.create_study(
        direction     = "maximize",
        sampler       = optuna.samplers.TPESampler(seed=SEED),
        study_name    = "dnn_fraud_auc",
        storage       = "sqlite:///optuna_dnn.db",
        load_if_exists= True,
    )
    study.optimize(objective, n_trials=N_TUNE_TRIALS, show_progress_bar=True)
    print(f"\nBest val AUC : {study.best_value:.4f}")
    print(f"Best params  : {study.best_params}")
    return study.best_params


# ── Data loading ──────────────────────────────────────────────────────────────
def load_data():
    print("Loading preprocessed data …")
    with open(DATA_DIR / "column_metadata.pkl", "rb") as f:
        meta = pickle.load(f)

    target      = meta["target"]
    cat_cols    = meta["cat_cols"]
    vocab_sizes = meta["vocab_sizes"]

    train = pd.read_csv(DATA_DIR / "train.csv")
    val   = pd.read_csv(DATA_DIR / "val.csv")
    test  = pd.read_csv(DATA_DIR / "test.csv")

    # Filter to cols present in the dataframe (after feature engineering)
    cat_cols    = [c for c in cat_cols if c in train.columns]
    vocab_sizes = {c: vocab_sizes[c] for c in cat_cols}

    # Everything that is not a cat col or the target is treated as numerical
    num_cols = [c for c in train.columns if c not in cat_cols and c != target]

    print(f"  Cat features : {len(cat_cols)}")
    print(f"  Num features : {len(num_cols)}")
    print(f"  Train rows   : {len(train):,}  | fraud rate: {train[target].mean():.3%}")
    print(f"  Val rows     : {len(val):,}  | fraud rate: {val[target].mean():.3%}")

    train_ds = FraudDataset(train, cat_cols, num_cols, target)
    val_ds   = FraudDataset(val,   cat_cols, num_cols, target)
    test_ds  = FraudDataset(test,  cat_cols, num_cols, target)

    return train_ds, val_ds, test_ds, vocab_sizes, len(num_cols)


# ── Main ──────────────────────────────────────────────────────────────────────
DEFAULT_HPARAMS = {
    "embed_dim"   : 16,
    "hidden_dim"  : 512,   # first block width; tapers 512 → 256 → 128
    "n_blocks"    : 3,
    "dropout"     : 0.3,
    "lr"          : 5e-4,
    "weight_decay": 1e-4,
    "focal_alpha" : 0.25,
    "focal_gamma" : 2.0,
}


def main(do_tune: bool):
    train_ds, val_ds, test_ds, vocab_sizes, n_num = load_data()

    if do_tune:
        print(f"\nStarting Optuna DNN sweep ({N_TUNE_TRIALS} trials) …")
        best_params = tune(train_ds, val_ds, vocab_sizes, n_num, DEVICE)
        hparams = {**DEFAULT_HPARAMS, **best_params}
    else:
        hparams = DEFAULT_HPARAMS
        print(f"\nUsing default hyperparams: {hparams}")

    print("\n── Training final model ─────────────────────────────────────────")
    model, best_val_auc, test_auc, test_probs = run_training(
        train_ds, val_ds, test_ds,
        vocab_sizes, n_num, hparams, DEVICE, verbose=True,
    )

    # Save model and predictions
    model_path = OUTPUT_DIR / "dnn_fraud.pt"
    torch.save({
        "model_state": model.state_dict(),
        "hparams"    : hparams,
        "vocab_sizes": vocab_sizes,
        "n_num"      : n_num,
    }, model_path)
    print(f"\nModel saved → {model_path}")

    pd.DataFrame({"fraud_prob": test_probs}).to_csv(
        OUTPUT_DIR / "test_predictions.csv", index=False
    )



DO_TUNE = False  # Set to True to run Optuna sweep first
main(do_tune=DO_TUNE)



Using device: mps
Loading preprocessed data …
  Cat features : 17
  Num features : 254
  Train rows   : 472,443  | fraud rate: 3.499%
  Val rows     : 29,516  | fraud rate: 3.500%

Using default hyperparams: {'embed_dim': 16, 'hidden_dim': 512, 'n_blocks': 3, 'dropout': 0.3, 'lr': 0.0005, 'weight_decay': 0.0001, 'focal_alpha': 0.25, 'focal_gamma': 2.0}

── Training final model ─────────────────────────────────────────
Trainable parameters: 868,317
Epoch 001 | loss 0.0622 | val AUC 0.8699 | val PR-AUC 0.4137 | 6.4s ✓
Epoch 002 | loss 0.0480 | val AUC 0.9013 | val PR-AUC 0.4896 | 5.2s ✓
Epoch 003 | loss 0.0404 | val AUC 0.9192 | val PR-AUC 0.5569 | 5.3s ✓
Epoch 004 | loss 0.0355 | val AUC 0.9260 | val PR-AUC 0.5938 | 5.6s ✓
Epoch 005 | loss 0.0323 | val AUC 0.9285 | val PR-AUC 0.6149 | 5.5s ✓
Epoch 006 | loss 0.0300 | val AUC 0.9326 | val PR-AUC 0.6376 | 5.5s ✓
Epoch 007 | loss 0.0280 | val AUC 0.9327 | val PR-AUC 0.6564 | 5.3s ✓
Epoch 008 | loss 0.0261 | val AUC 0.9368 | val PR-AUC 0.67

In [35]:
"""
DNN Fraud Detection — Embedding + MLP (PyTorch)
================================================
Architecture:
  • Categorical cols  → individual Embedding layers
                        → EmbedAttention (self-attention over cat embeddings)
                        → concatenated
  • Numerical cols    → passed directly (already scaled)
  • Concat            → BatchNorm → MLP with TAPERED residual skip connections
                        (512 → 256 → 128) → Sigmoid output

Improvements over v1:
  • EmbedAttention: self-attention over categorical embeddings before concat
  • Stochastic depth in residual blocks
  • SiLU activations (replaces GELU)
  • LR warm-up + cosine annealing via SequentialLR
  • Label smoothing in FocalLoss
  • Sampler and focal loss de-conflicted (alpha auto-set when sampler is on)
  • Threshold sweep on VAL set, applied to TEST set (no leakage)
  • Optuna sweep extended: label_smoothing, warmup_epochs, embed_attn,
    drop_path_rate — and MedianPruner to kill bad trials early
  • PATIENCE raised to 12

Device priority: CUDA → MPS (Apple Silicon) → CPU

Expects preprocessed outputs from the pipeline:
  preprocessed/train.csv
  preprocessed/val.csv
  preprocessed/test.csv
  preprocessed/column_metadata.pkl

Usage:
  python dnn_fraud.py              # train with default hyperparams
  python dnn_fraud.py --tune       # Optuna sweep over arch hyperparams
"""

import argparse
import pickle
import time
from pathlib import Path

import numpy as np
import optuna
from optuna.pruners import MedianPruner
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    f1_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

# ── Config ────────────────────────────────────────────────────────────────────

DATA_DIR      = Path("preprocessed")
OUTPUT_DIR    = Path("models/dnn")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED          = 42
BATCH_SIZE    = 4096
MAX_EPOCHS    = 50
PATIENCE      = 12       # raised from 7 — cosine schedule needs room to breathe
N_TUNE_TRIALS = 75       # raised from 30; TPE needs ~50-100 for 10+ dims

torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────────

def get_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
    elif torch.backends.mps.is_available():
        dev = torch.device("mps")
    else:
        dev = torch.device("cpu")
    print(f"Using device: {dev}")
    return dev

DEVICE = get_device()

# ── Dataset ───────────────────────────────────────────────────────────────────

class FraudDataset(Dataset):
    """
    Splits columns into:
      cat_data  — integer indices for embedding lookup  (LongTensor)
      num_data  — scaled floats                          (FloatTensor)
      labels    — 0/1                                    (FloatTensor)
    """
    def __init__(self, df, cat_cols, num_cols, target):
        self.cat = torch.tensor(df[cat_cols].values, dtype=torch.long)
        self.num = torch.tensor(df[num_cols].values, dtype=torch.float32)
        self.y   = torch.tensor(df[target].values,   dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.cat[idx], self.num[idx], self.y[idx]


def make_sampler(y_train_np):
    """WeightedRandomSampler → each mini-batch ~50/50 fraud/legit."""
    fraud_rate = y_train_np.mean()
    w_fraud    = 1.0 / fraud_rate
    w_legit    = 1.0 / (1.0 - fraud_rate)
    weights    = np.where(y_train_np == 1, w_fraud, w_legit)
    return WeightedRandomSampler(
        weights     = torch.DoubleTensor(weights),
        num_samples = len(weights),
        replacement = True,
    )

# ── Model components ──────────────────────────────────────────────────────────

class EmbedAttention(nn.Module):
    """
    Self-attention over the set of categorical embeddings.

    Allows the model to learn joint signals across categoricals
    before they're concatenated with numerical features.
    E.g. 'new device + new email + high amount' is more suspicious
    than any single feature alone.

    Input shape:  (B, n_cats, embed_dim)
    Output shape: (B, n_cats, embed_dim)  [residual-normed]
    """
    def __init__(self, embed_dim: int, n_heads: int = 4):
        super().__init__()
        # n_heads must divide embed_dim; clamp to safe value
        n_heads = max(1, min(n_heads, embed_dim // 4))
        self.attn = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True,
                                          dropout=0.1)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, embeds: torch.Tensor) -> torch.Tensor:
        out, _ = self.attn(embeds, embeds, embeds)
        return self.norm(embeds + out)


class TaperedResidualBlock(nn.Module):
    """
    Residual block that projects from in_dim → out_dim.

    When in_dim != out_dim a linear shortcut aligns the skip connection.
    Stochastic depth (drop_path_rate) randomly skips the entire block
    during training, acting as a strong structural regulariser.
    """
    def __init__(self, in_dim: int, out_dim: int, dropout: float,
                 drop_path_rate: float = 0.0):
        super().__init__()
        self.drop_path_rate = drop_path_rate

        self.block = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.SiLU(),                          # SiLU/Swish instead of GELU
            nn.Dropout(dropout),
            nn.Linear(out_dim, out_dim),
            nn.BatchNorm1d(out_dim),
        )
        self.shortcut = (
            nn.Linear(in_dim, out_dim, bias=False)
            if in_dim != out_dim else nn.Identity()
        )
        self.act = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shortcut = self.shortcut(x)
        # Stochastic depth: skip the residual branch entirely with p=drop_path_rate
        if self.training and self.drop_path_rate > 0.0:
            if torch.rand(1).item() < self.drop_path_rate:
                return self.act(shortcut)
        return self.act(shortcut + self.block(x))


class EmbeddingMLP(nn.Module):
    """
    Embedding + self-attention + MLP fraud classifier.

    Pipeline
    --------
    1. Each categorical column → its own Embedding layer.
    2. All embeddings stacked → EmbedAttention (optional, controlled by
       use_embed_attn flag) → flattened.
    3. Concat with scaled numerical features.
    4. Input projection → hidden_dim.
    5. Tapered residual blocks: hidden_dim → hidden_dim//2 → … (min 64).
    6. Output head → scalar logit.

    Parameters
    ----------
    vocab_sizes    : dict  {col_name: vocab_size}
    num_features   : int   number of numerical input features
    embed_dim      : int   max embedding dimension per column
    hidden_dim     : int   width of the FIRST residual block
    n_blocks       : int   number of residual blocks
    dropout        : float dropout rate
    drop_path_rate : float stochastic depth probability per block
    use_embed_attn : bool  whether to apply self-attention over embeddings
    """

    def __init__(
        self,
        vocab_sizes    : dict,
        num_features   : int,
        embed_dim      : int   = 16,
        hidden_dim     : int   = 512,
        n_blocks       : int   = 3,
        dropout        : float = 0.3,
        drop_path_rate : float = 0.05,
        use_embed_attn : bool  = True,
    ):
        super().__init__()
        self.cat_cols      = list(vocab_sizes.keys())
        self.use_embed_attn = use_embed_attn

        # Per-column embedding dims (rule of thumb)
        col_dims = {
            col: min(embed_dim, max(4, vocab_sizes[col] // 2))
            for col in self.cat_cols
        }
        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_sizes[col], col_dims[col])
            for col in self.cat_cols
        ])

        embed_total = sum(col_dims.values())

        # Self-attention requires all embeddings to share a common dim;
        # we only apply it when embed_dim is uniform (i.e. all cols hit the cap).
        # If dims vary, we skip attn gracefully.
        unique_dims = set(col_dims.values())
        if use_embed_attn and len(unique_dims) == 1 and len(self.cat_cols) > 1:
            common_dim = unique_dims.pop()
            self.embed_attn = EmbedAttention(common_dim)
        else:
            self.embed_attn  = None
            self.use_embed_attn = False

        input_dim = embed_total + num_features

        # Input projection → first block width
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(dropout),
        )

        # Tapered residual blocks
        dims = []
        w = hidden_dim
        for _ in range(n_blocks):
            next_w = max(w // 2, 64)
            dims.append((w, next_w))
            w = next_w

        self.blocks = nn.ModuleList([
            TaperedResidualBlock(in_d, out_d, dropout, drop_path_rate)
            for in_d, out_d in dims
        ])

        final_dim = dims[-1][1]

        # Output head
        self.head = nn.Sequential(
            nn.Linear(final_dim, final_dim // 2),
            nn.SiLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(final_dim // 2, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)

    def forward(self, cat: torch.Tensor, num: torch.Tensor) -> torch.Tensor:
        embeds = [emb(cat[:, i]) for i, emb in enumerate(self.embeddings)]

        if self.use_embed_attn and self.embed_attn is not None:
            # Stack → (B, n_cats, embed_dim) → attention → flatten
            stacked  = torch.stack(embeds, dim=1)      # (B, n_cats, D)
            attended = self.embed_attn(stacked)        # (B, n_cats, D)
            embeds   = [attended[:, i, :] for i in range(attended.size(1))]

        x = torch.cat(embeds + [num], dim=1)
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x).squeeze(1)                 # raw logits, shape (B,)


# ── Focal Loss ────────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    """
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    Focuses training on hard, misclassified examples.

    label_smoothing: mix 0/1 targets with uniform noise to prevent
    overconfidence on noisy fraud labels (a small but reliable gain).

    Note on sampler interaction:
      If WeightedRandomSampler is active the batch class distribution is
      already ~50/50.  In that case set alpha close to 0.5 (or let Optuna
      find it); the default 0.25 is calibrated for imbalanced batches.
    """
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0,
                 label_smoothing: float = 0.0):
        super().__init__()
        self.alpha           = alpha
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # Apply label smoothing
        if self.label_smoothing > 0.0:
            targets = targets * (1.0 - self.label_smoothing) + 0.5 * self.label_smoothing

        bce  = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        prob = torch.sigmoid(logits)
        pt   = torch.where(targets >= 0.5, prob, 1.0 - prob)
        at   = torch.where(
            targets >= 0.5,
            torch.full_like(pt, self.alpha),
            torch.full_like(pt, 1.0 - self.alpha),
        )
        return (at * (1.0 - pt) ** self.gamma * bce).mean()


# ── LR schedule helpers ───────────────────────────────────────────────────────

def build_scheduler(optimiser, warmup_epochs: int, total_epochs: int):
    """
    Linear warm-up for `warmup_epochs`, then cosine decay to eta_min=1e-6.

    A large batch size (4096) benefits from a gentle ramp-up so the model
    doesn't take huge gradient steps before the batch statistics stabilise.
    """
    if warmup_epochs > 0:
        warmup = torch.optim.lr_scheduler.LinearLR(
            optimiser,
            start_factor = 0.1,
            end_factor   = 1.0,
            total_iters  = warmup_epochs,
        )
        cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimiser,
            T_max   = max(1, total_epochs - warmup_epochs),
            eta_min = 1e-6,
        )
        return torch.optim.lr_scheduler.SequentialLR(
            optimiser,
            schedulers  = [warmup, cosine],
            milestones  = [warmup_epochs],
        )
    else:
        return torch.optim.lr_scheduler.CosineAnnealingLR(
            optimiser, T_max=total_epochs, eta_min=1e-6
        )


# ── Train / eval loops ────────────────────────────────────────────────────────

def train_epoch(model, loader, optimiser, criterion, device):
    model.train()
    total_loss = 0.0
    for cat, num, y in loader:
        cat, num, y = cat.to(device), num.to(device), y.to(device)
        optimiser.zero_grad()
        loss = criterion(model(cat, num), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_loader(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for cat, num, y in loader:
        probs = torch.sigmoid(model(cat.to(device), num.to(device))).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    return roc_auc_score(labels, probs), average_precision_score(labels, probs), probs, labels


# ── Full training run ─────────────────────────────────────────────────────────

def run_training(train_ds, val_ds, test_ds,
                 vocab_sizes, n_num, hparams, device, verbose=True,
                 trial=None):
    """
    Parameters
    ----------
    trial : optuna.Trial or None
        If provided, reports intermediate val AUC values for Optuna pruning.
    """
    pin = device.type == "cuda"

    train_dl = DataLoader(
        train_ds, batch_size=BATCH_SIZE,
        sampler=make_sampler(train_ds.y.numpy()),
        num_workers=0, pin_memory=pin,
    )
    val_dl  = DataLoader(val_ds,  batch_size=BATCH_SIZE * 2,
                         shuffle=False, num_workers=0)
    test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE * 2,
                         shuffle=False, num_workers=0)

    model = EmbeddingMLP(
        vocab_sizes    = vocab_sizes,
        num_features   = n_num,
        embed_dim      = hparams["embed_dim"],
        hidden_dim     = hparams["hidden_dim"],
        n_blocks       = hparams["n_blocks"],
        dropout        = hparams["dropout"],
        drop_path_rate = hparams["drop_path_rate"],
        use_embed_attn = hparams["use_embed_attn"],
    ).to(device)

    if verbose:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Trainable parameters: {n_params:,}")

    criterion = FocalLoss(
        alpha           = hparams["focal_alpha"],
        gamma           = hparams["focal_gamma"],
        label_smoothing = hparams["label_smoothing"],
    )

    optimiser = torch.optim.AdamW(
        model.parameters(),
        lr=hparams["lr"], weight_decay=hparams["weight_decay"],
    )

    scheduler = build_scheduler(
        optimiser,
        warmup_epochs = hparams["warmup_epochs"],
        total_epochs  = MAX_EPOCHS,
    )

    best_auc   = 0.0
    best_state = None
    patience_n = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        t0         = time.time()
        train_loss = train_epoch(model, train_dl, optimiser, criterion, device)
        val_auc, val_pr, _, _ = eval_loader(model, val_dl, device)
        scheduler.step()

        # Optuna pruning hook
        if trial is not None:
            trial.report(val_auc, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        improved = val_auc > best_auc
        if improved:
            best_auc   = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_n = 0
        else:
            patience_n += 1

        if verbose:
            tick = " ✓" if improved else ""
            print(
                f"Epoch {epoch:03d} | loss {train_loss:.4f} | "
                f"val AUC {val_auc:.4f} | val PR-AUC {val_pr:.4f} | "
                f"{time.time()-t0:.1f}s{tick}"
            )

        if patience_n >= PATIENCE:
            if verbose:
                print(f"Early stopping triggered at epoch {epoch}.")
            break

    # Restore best checkpoint
    model.load_state_dict(best_state)

    # ── Threshold selection on VAL set (no test leakage) ──────────────────────
    _, _, val_probs, val_labels = eval_loader(model, val_dl, device)
    thresholds = np.linspace(0.05, 0.95, 91)
    f1s        = [f1_score(val_labels, val_probs >= t, zero_division=0)
                  for t in thresholds]
    best_t     = thresholds[np.argmax(f1s)]

    # ── Evaluate on test set ──────────────────────────────────────────────────
    test_auc, test_pr, test_probs, test_labels = eval_loader(model, test_dl, device)

    if verbose:
        print(f"\nBest val  AUC  : {best_auc:.4f}")
        print(f"Test      AUC  : {test_auc:.4f}")
        print(f"Test   PR-AUC  : {test_pr:.4f}")
        print(f"Best threshold (F1 on val set): {best_t:.2f}")

        # Precision @ recall targets — more useful than raw F1 in production
        from sklearn.metrics import precision_recall_curve
        prec, rec, thr = precision_recall_curve(test_labels, test_probs)
        for recall_target in [0.70, 0.80, 0.90]:
            mask = rec >= recall_target
            if mask.any():
                p_at_r = prec[mask][-1]
                t_at_r = thr[mask[:-1]][-1] if mask[:-1].any() else best_t
                print(f"  Precision @ recall≥{recall_target:.0%}: {p_at_r:.4f}  (threshold {t_at_r:.3f})")

        print(classification_report(
            test_labels, test_probs >= best_t,
            target_names=["legit", "fraud"],
        ))

    return model, best_auc, test_auc, test_probs, best_t


# ── Optuna sweep ──────────────────────────────────────────────────────────────

def tune(train_ds, val_ds, vocab_sizes, n_num, device):
    def objective(trial):
        hp = {
            "embed_dim"      : trial.suggest_categorical("embed_dim",   [8, 16, 32]),
            "hidden_dim"     : trial.suggest_categorical("hidden_dim",  [256, 512, 1024]),
            "n_blocks"       : trial.suggest_int("n_blocks",            2, 5),
            "dropout"        : trial.suggest_float("dropout",           0.1, 0.5),
            "drop_path_rate" : trial.suggest_float("drop_path_rate",    0.0, 0.15),
            "use_embed_attn" : trial.suggest_categorical("use_embed_attn", [True, False]),
            "lr"             : trial.suggest_float("lr",                1e-4, 3e-3, log=True),
            "weight_decay"   : trial.suggest_float("weight_decay",      1e-5, 1e-2, log=True),
            # When sampler is active, alpha should be near 0.5; allow full range
            "focal_alpha"    : trial.suggest_float("focal_alpha",       0.1, 0.5),
            "focal_gamma"    : trial.suggest_float("focal_gamma",       1.0, 5.0),
            "label_smoothing": trial.suggest_float("label_smoothing",   0.0, 0.1),
            "warmup_epochs"  : trial.suggest_int("warmup_epochs",       0, 5),
        }

        _, val_auc, _, _, _ = run_training(
            train_ds, val_ds, val_ds,
            vocab_sizes, n_num, hp, device,
            verbose=False, trial=trial,
        )
        return val_auc

    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=5)
    study  = optuna.create_study(
        direction      = "maximize",
        sampler        = optuna.samplers.TPESampler(seed=SEED),
        pruner         = pruner,
        study_name     = "dnn_fraud_auc_v2",
        storage        = "sqlite:///optuna_dnn.db",
        load_if_exists = True,
    )
    study.optimize(objective, n_trials=N_TUNE_TRIALS, show_progress_bar=True)

    print(f"\nBest val AUC : {study.best_value:.4f}")
    print(f"Best params  : {study.best_params}")
    return study.best_params


# ── Data loading ──────────────────────────────────────────────────────────────

def load_data():
    print("Loading preprocessed data ...")

    with open(DATA_DIR / "column_metadata.pkl", "rb") as f:
        meta = pickle.load(f)

    target      = meta["target"]
    cat_cols    = meta["cat_cols"]
    vocab_sizes = meta["vocab_sizes"]

    train = pd.read_csv(DATA_DIR / "train.csv")
    val   = pd.read_csv(DATA_DIR / "val.csv")
    test  = pd.read_csv(DATA_DIR / "test.csv")

    # Filter to cols present in the dataframe
    cat_cols    = [c for c in cat_cols    if c in train.columns]
    vocab_sizes = {c: vocab_sizes[c]      for c in cat_cols}
    num_cols    = [c for c in train.columns
                   if c not in cat_cols and c != target]

    print(f"  Cat features : {len(cat_cols)}")
    print(f"  Num features : {len(num_cols)}")
    print(f"  Train rows   : {len(train):,}  | fraud rate: {train[target].mean():.3%}")
    print(f"  Val rows     : {len(val):,}  | fraud rate: {val[target].mean():.3%}")
    print(f"  Test rows    : {len(test):,}  | fraud rate: {test[target].mean():.3%}")

    train_ds = FraudDataset(train, cat_cols, num_cols, target)
    val_ds   = FraudDataset(val,   cat_cols, num_cols, target)
    test_ds  = FraudDataset(test,  cat_cols, num_cols, target)

    return train_ds, val_ds, test_ds, vocab_sizes, len(num_cols)


# ── Main ──────────────────────────────────────────────────────────────────────

DEFAULT_HPARAMS = {
    "embed_dim"      : 16,
    "hidden_dim"     : 512,
    "n_blocks"       : 3,
    "dropout"        : 0.3,
    "drop_path_rate" : 0.05,
    "use_embed_attn" : True,
    "lr"             : 5e-4,
    "weight_decay"   : 1e-4,
    # With sampler making batches 50/50, alpha closer to 0.5 is more appropriate
    "focal_alpha"    : 0.4,
    "focal_gamma"    : 2.0,
    "label_smoothing": 0.03,
    "warmup_epochs"  : 3,
}


def main(do_tune: bool):
    train_ds, val_ds, test_ds, vocab_sizes, n_num = load_data()

    if do_tune:
        print(f"\nStarting Optuna DNN sweep ({N_TUNE_TRIALS} trials) ...")
        best_params = tune(train_ds, val_ds, vocab_sizes, n_num, DEVICE)
        hparams     = {**DEFAULT_HPARAMS, **best_params}
    else:
        hparams = DEFAULT_HPARAMS
        print(f"\nUsing default hyperparams: {hparams}")

    print("\n── Training final model ─────────────────────────────────────────")
    model, best_val_auc, test_auc, test_probs, best_t = run_training(
        train_ds, val_ds, test_ds,
        vocab_sizes, n_num, hparams, DEVICE, verbose=True,
    )

    # Save model, threshold, and predictions
    model_path = OUTPUT_DIR / "dnn_fraud.pt"
    torch.save({
        "model_state"  : model.state_dict(),
        "hparams"      : hparams,
        "vocab_sizes"  : vocab_sizes,
        "n_num"        : n_num,
        "best_threshold": best_t,
    }, model_path)
    print(f"\nModel saved → {model_path}")

    pd.DataFrame({
        "fraud_prob"   : test_probs,
        "fraud_pred"   : (test_probs >= best_t).astype(int),
    }).to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--tune", action="store_true",
                        help="Run Optuna hyperparameter sweep before final training")
    args, unknown = parser.parse_known_args()
    main(do_tune=args.tune)

Using device: mps
Loading preprocessed data ...
  Cat features : 17
  Num features : 254
  Train rows   : 472,443  | fraud rate: 3.499%
  Val rows     : 29,516  | fraud rate: 3.500%
  Test rows    : 88,581  | fraud rate: 3.498%

Using default hyperparams: {'embed_dim': 16, 'hidden_dim': 512, 'n_blocks': 3, 'dropout': 0.3, 'drop_path_rate': 0.05, 'use_embed_attn': True, 'lr': 0.0005, 'weight_decay': 0.0001, 'focal_alpha': 0.4, 'focal_gamma': 2.0, 'label_smoothing': 0.03, 'warmup_epochs': 3}

── Training final model ─────────────────────────────────────────
Trainable parameters: 868,317
Epoch 001 | loss 0.1005 | val AUC 0.8326 | val PR-AUC 0.3021 | 5.5s ✓
Epoch 002 | loss 0.0687 | val AUC 0.8616 | val PR-AUC 0.3884 | 5.3s ✓


/Users/anbu/Library/Python/3.9/lib/python/site-packages/torch/optim/lr_scheduler.py:209: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Epoch 003 | loss 0.0607 | val AUC 0.8845 | val PR-AUC 0.4479 | 5.5s ✓
Epoch 004 | loss 0.0519 | val AUC 0.9090 | val PR-AUC 0.5186 | 5.3s ✓
Epoch 005 | loss 0.0444 | val AUC 0.9170 | val PR-AUC 0.5550 | 5.8s ✓
Epoch 006 | loss 0.0410 | val AUC 0.9214 | val PR-AUC 0.5827 | 5.7s ✓
Epoch 007 | loss 0.0385 | val AUC 0.9241 | val PR-AUC 0.6025 | 5.6s ✓
Epoch 008 | loss 0.0363 | val AUC 0.9285 | val PR-AUC 0.6260 | 5.6s ✓
Epoch 009 | loss 0.0341 | val AUC 0.9309 | val PR-AUC 0.6388 | 5.5s ✓
Epoch 010 | loss 0.0323 | val AUC 0.9320 | val PR-AUC 0.6533 | 5.6s ✓
Epoch 011 | loss 0.0307 | val AUC 0.9344 | val PR-AUC 0.6676 | 5.7s ✓
Epoch 012 | loss 0.0293 | val AUC 0.9348 | val PR-AUC 0.6778 | 5.6s ✓
Epoch 013 | loss 0.0279 | val AUC 0.9370 | val PR-AUC 0.6899 | 5.5s ✓
Epoch 014 | loss 0.0262 | val AUC 0.9384 | val PR-AUC 0.6983 | 5.6s ✓
Epoch 015 | loss 0.0249 | val AUC 0.9399 | val PR-AUC 0.7078 | 5.6s ✓
Epoch 016 | loss 0.0240 | val AUC 0.9380 | val PR-AUC 0.7107 | 5.5s
Epoch 017 | loss 0.022

### Selected Model from the above 3

In [ ]:
"""
DNN Fraud Detection — Embedding + MLP (PyTorch)
================================================
Architecture:
  • Categorical cols  → individual Embedding layers → concatenated
  • Numerical cols    → passed directly (already scaled)
  • Concat            → BatchNorm → MLP with TAPERED residual skip connections
                        (512 → 256 → 128) → Sigmoid output

Device priority: CUDA → MPS (Apple Silicon) → CPU

Expects preprocessed outputs from the pipeline:
  preprocessed/train.csv
  preprocessed/val.csv
  preprocessed/test.csv
  preprocessed/column_metadata.pkl

Usage:
  python dnn_fraud.py              # train with default hyperparams
  python dnn_fraud.py --tune       # Optuna sweep over arch hyperparams
"""

import argparse
import pickle
import time
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR      = Path("preprocessed")
OUTPUT_DIR    = Path("models/dnn")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED          = 42
BATCH_SIZE    = 4096
MAX_EPOCHS    = 50
PATIENCE      = 7        # early stopping patience on val AUC
N_TUNE_TRIALS = 30       # Optuna trials when --tune is passed

torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
def get_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
    elif torch.backends.mps.is_available():
        dev = torch.device("mps")
    else:
        dev = torch.device("cpu")
    print(f"Using device: {dev}")
    return dev

DEVICE = get_device()


# ── Dataset ───────────────────────────────────────────────────────────────────
class FraudDataset(Dataset):
    """
    Splits columns into:
      cat_data  — integer indices for embedding lookup  (LongTensor)
      num_data  — scaled floats                          (FloatTensor)
      labels    — 0/1                                    (FloatTensor)
    """
    def __init__(self, df, cat_cols, num_cols, target):
        self.cat = torch.tensor(df[cat_cols].values, dtype=torch.long)
        self.num = torch.tensor(df[num_cols].values, dtype=torch.float32)
        self.y   = torch.tensor(df[target].values,   dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.cat[idx], self.num[idx], self.y[idx]


def make_sampler(y_train_np):
    """WeightedRandomSampler → each mini-batch ~50/50 fraud/legit."""
    fraud_rate = y_train_np.mean()
    w_fraud    = 1.0 / fraud_rate
    w_legit    = 1.0 / (1.0 - fraud_rate)
    weights    = np.where(y_train_np == 1, w_fraud, w_legit)
    return WeightedRandomSampler(
        weights     = torch.DoubleTensor(weights),
        num_samples = len(weights),
        replacement = True,
    )


# ── Model ─────────────────────────────────────────────────────────────────────
class TaperedResidualBlock(nn.Module):
    """
    Residual block that projects from in_dim → out_dim.
    When in_dim != out_dim a linear shortcut aligns the skip connection,
    so gradients still flow cleanly at every width transition.
    """
    def __init__(self, in_dim: int, out_dim: int, dropout: float):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim, out_dim),
            nn.BatchNorm1d(out_dim),
        )
        # Shortcut: project only when widths differ
        self.shortcut = (
            nn.Linear(in_dim, out_dim, bias=False)
            if in_dim != out_dim else nn.Identity()
        )
        self.act = nn.GELU()

    def forward(self, x):
        return self.act(self.shortcut(x) + self.block(x))


class EmbeddingMLP(nn.Module):
    """
    Embedding + MLP fraud classifier with tapered residual blocks.

    Each categorical column gets its own Embedding layer.
    Embedding dim uses the rule-of-thumb: min(embed_dim, max(4, vocab//2)).
    All embeddings + numerical features are concatenated, projected to
    hidden_dim (= first block width), then passed through a stack of
    tapered residual blocks that halve the width each time.

    Block widths: hidden_dim → hidden_dim//2 → hidden_dim//4
    Default with hidden_dim=512: 512 → 256 → 128

    Parameters
    ----------
    vocab_sizes  : dict  {col_name: vocab_size}
    num_features : int   number of numerical input features
    embed_dim    : int   max embedding dimension per column
    hidden_dim   : int   width of the FIRST residual block (should be 512)
    n_blocks     : int   number of residual blocks (3 recommended for tapering)
    dropout      : float dropout rate
    """
    def __init__(
        self,
        vocab_sizes : dict,
        num_features: int,
        embed_dim   : int   = 16,
        hidden_dim  : int   = 512,
        n_blocks    : int   = 3,
        dropout     : float = 0.3,
    ):
        super().__init__()
        self.cat_cols = list(vocab_sizes.keys())

        # Per-column embedding dims
        col_dims = {
            col: min(embed_dim, max(4, vocab_sizes[col] // 2))
            for col in self.cat_cols
        }
        self.embeddings = nn.ModuleList([
            nn.Embedding(vocab_sizes[col], col_dims[col])
            for col in self.cat_cols
        ])

        embed_total = sum(col_dims.values())
        input_dim   = embed_total + num_features

        # Input projection → first block width
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Tapered residual blocks: each block halves the width
        # e.g. hidden_dim=512 → [512→256, 256→128, 128→64]
        # but we clamp so width never falls below 64
        dims = []
        w = hidden_dim
        for _ in range(n_blocks):
            next_w = max(w // 2, 64)
            dims.append((w, next_w))
            w = next_w

        self.blocks = nn.ModuleList([
            TaperedResidualBlock(in_d, out_d, dropout)
            for in_d, out_d in dims
        ])

        final_dim = dims[-1][1]   # width after last block

        # Output head
        self.head = nn.Sequential(
            nn.Linear(final_dim, final_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(final_dim // 2, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)

    def forward(self, cat, num):
        embeds = [emb(cat[:, i]) for i, emb in enumerate(self.embeddings)]
        x = torch.cat(embeds + [num], dim=1)
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x).squeeze(1)   # raw logits, shape (B,)


# ── Focal Loss ────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    Focuses training on hard, misclassified examples.
    Helps with the heavy class imbalance in fraud datasets.
    """
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce  = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        prob = torch.sigmoid(logits)
        pt   = torch.where(targets == 1, prob, 1.0 - prob)
        at   = torch.where(
            targets == 1,
            torch.full_like(pt, self.alpha),
            torch.full_like(pt, 1.0 - self.alpha),
        )
        return (at * (1.0 - pt) ** self.gamma * bce).mean()


# ── Train / eval loops ────────────────────────────────────────────────────────
def train_epoch(model, loader, optimiser, criterion, device):
    model.train()
    total_loss = 0.0
    for cat, num, y in loader:
        cat, num, y = cat.to(device), num.to(device), y.to(device)
        optimiser.zero_grad()
        loss = criterion(model(cat, num), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def eval_loader(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for cat, num, y in loader:
        probs = torch.sigmoid(model(cat.to(device), num.to(device))).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    return roc_auc_score(labels, probs), average_precision_score(labels, probs), probs, labels


# ── Full training run ─────────────────────────────────────────────────────────
def run_training(train_ds, val_ds, test_ds,
                 vocab_sizes, n_num, hparams, device, verbose=True):

    # pin_memory causes issues on MPS
    pin = device.type == "cuda"

    train_dl = DataLoader(
        train_ds, batch_size=BATCH_SIZE,
        sampler=make_sampler(train_ds.y.numpy()),
        num_workers=0, pin_memory=pin,
    )
    val_dl = DataLoader(val_ds,  batch_size=BATCH_SIZE * 2,
                        shuffle=False, num_workers=0)
    test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE * 2,
                         shuffle=False, num_workers=0)

    model = EmbeddingMLP(
        vocab_sizes  = vocab_sizes,
        num_features = n_num,
        embed_dim    = hparams["embed_dim"],
        hidden_dim   = hparams["hidden_dim"],
        n_blocks     = hparams["n_blocks"],
        dropout      = hparams["dropout"],
    ).to(device)

    if verbose:
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Trainable parameters: {n_params:,}")

    criterion = FocalLoss(alpha=hparams["focal_alpha"], gamma=hparams["focal_gamma"])
    optimiser = torch.optim.AdamW(
        model.parameters(),
        lr=hparams["lr"], weight_decay=hparams["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimiser, T_max=MAX_EPOCHS, eta_min=1e-6
    )

    best_auc   = 0.0
    best_state = None
    patience_n = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        t0         = time.time()
        train_loss = train_epoch(model, train_dl, optimiser, criterion, device)
        val_auc, val_pr, _, _ = eval_loader(model, val_dl, device)
        scheduler.step()

        improved = val_auc > best_auc
        if improved:
            best_auc   = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_n = 0
        else:
            patience_n += 1

        if verbose:
            tick = " ✓" if improved else ""
            print(
                f"Epoch {epoch:03d} | loss {train_loss:.4f} | "
                f"val AUC {val_auc:.4f} | val PR-AUC {val_pr:.4f} | "
                f"{time.time()-t0:.1f}s{tick}"
            )

        if patience_n >= PATIENCE:
            if verbose:
                print(f"Early stopping triggered at epoch {epoch}.")
            break

    # Restore best checkpoint and evaluate on test
    model.load_state_dict(best_state)
    test_auc, test_pr, test_probs, test_labels = eval_loader(model, test_dl, device)

    if verbose:
        print(f"\nBest val  AUC  : {best_auc:.4f}")
        print(f"Test      AUC  : {test_auc:.4f}")
        print(f"Test   PR-AUC  : {test_pr:.4f}")
        # Threshold sweep for best F1
        from sklearn.metrics import f1_score
        thresholds = np.linspace(0.05, 0.95, 91)
        f1s        = [f1_score(test_labels, test_probs >= t, zero_division=0)
                      for t in thresholds]
        best_t     = thresholds[np.argmax(f1s)]
        print(f"Best threshold (F1 on test): {best_t:.2f}")
        print(classification_report(
            test_labels, test_probs >= best_t,
            target_names=["legit", "fraud"],
        ))

    return model, best_auc, test_auc, test_probs


# ── Optuna sweep ──────────────────────────────────────────────────────────────
def tune(train_ds, val_ds, vocab_sizes, n_num, device):
    def objective(trial):
        hp = {
            "embed_dim"   : trial.suggest_categorical("embed_dim",  [8, 16, 32]),
            # hidden_dim is now the FIRST (widest) block; taper does the rest
            "hidden_dim"  : trial.suggest_categorical("hidden_dim", [256, 512, 1024]),
            "n_blocks"    : trial.suggest_int("n_blocks", 2, 5),
            "dropout"     : trial.suggest_float("dropout", 0.1, 0.5),
            "lr"          : trial.suggest_float("lr", 1e-4, 3e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
            "focal_alpha" : trial.suggest_float("focal_alpha", 0.1, 0.5),
            "focal_gamma" : trial.suggest_float("focal_gamma", 1.0, 4.0),
        }
        _, val_auc, _, _ = run_training(
            train_ds, val_ds, val_ds,
            vocab_sizes, n_num, hp, device, verbose=False,
        )
        return val_auc

    study = optuna.create_study(
        direction     = "maximize",
        sampler       = optuna.samplers.TPESampler(seed=SEED),
        study_name    = "dnn_fraud_auc",
        storage       = "sqlite:///optuna_dnn.db",
        load_if_exists= True,
    )
    study.optimize(objective, n_trials=N_TUNE_TRIALS, show_progress_bar=True)
    print(f"\nBest val AUC : {study.best_value:.4f}")
    print(f"Best params  : {study.best_params}")
    return study.best_params


# ── Data loading ──────────────────────────────────────────────────────────────
def load_data():
    print("Loading preprocessed data …")
    with open(DATA_DIR / "column_metadata.pkl", "rb") as f:
        meta = pickle.load(f)

    target      = meta["target"]
    cat_cols    = meta["cat_cols"]
    vocab_sizes = meta["vocab_sizes"]

    train = pd.read_csv(DATA_DIR / "train.csv")
    val   = pd.read_csv(DATA_DIR / "val.csv")
    test  = pd.read_csv(DATA_DIR / "test.csv")

    # Filter to cols present in the dataframe (after feature engineering)
    cat_cols    = [c for c in cat_cols if c in train.columns]
    vocab_sizes = {c: vocab_sizes[c] for c in cat_cols}

    # Everything that is not a cat col or the target is treated as numerical
    num_cols = [c for c in train.columns if c not in cat_cols and c != target]

    print(f"  Cat features : {len(cat_cols)}")
    print(f"  Num features : {len(num_cols)}")
    print(f"  Train rows   : {len(train):,}  | fraud rate: {train[target].mean():.3%}")
    print(f"  Val rows     : {len(val):,}  | fraud rate: {val[target].mean():.3%}")

    train_ds = FraudDataset(train, cat_cols, num_cols, target)
    val_ds   = FraudDataset(val,   cat_cols, num_cols, target)
    test_ds  = FraudDataset(test,  cat_cols, num_cols, target)

    return train_ds, val_ds, test_ds, vocab_sizes, len(num_cols)


# ── Main ──────────────────────────────────────────────────────────────────────
DEFAULT_HPARAMS = {
    "embed_dim"   : 16,
    "hidden_dim"  : 512,   # first block width; tapers 512 → 256 → 128
    "n_blocks"    : 3,
    "dropout"     : 0.3,
    "lr"          : 5e-4,
    "weight_decay": 1e-4,
    "focal_alpha" : 0.25,
    "focal_gamma" : 2.0,
}


def main(do_tune: bool):
    train_ds, val_ds, test_ds, vocab_sizes, n_num = load_data()

    if do_tune:
        print(f"\nStarting Optuna DNN sweep ({N_TUNE_TRIALS} trials) …")
        best_params = tune(train_ds, val_ds, vocab_sizes, n_num, DEVICE)
        hparams = {**DEFAULT_HPARAMS, **best_params}
    else:
        hparams = DEFAULT_HPARAMS
        print(f"\nUsing default hyperparams: {hparams}")

    print("\n── Training final model ─────────────────────────────────────────")
    model, best_val_auc, test_auc, test_probs = run_training(
        train_ds, val_ds, test_ds,
        vocab_sizes, n_num, hparams, DEVICE, verbose=True,
    )

    # Save model and predictions
    model_path = OUTPUT_DIR / "dnn_fraud.pt"
    torch.save({
        "model_state": model.state_dict(),
        "hparams"    : hparams,
        "vocab_sizes": vocab_sizes,
        "n_num"      : n_num,
    }, model_path)
    print(f"\nModel saved → {model_path}")

    pd.DataFrame({"fraud_prob": test_probs}).to_csv(
        OUTPUT_DIR / "test_predictions.csv", index=False
    )



DO_TUNE = False  # Set to True to run Optuna sweep first
main(do_tune=DO_TUNE)

